# Baseline 与 RMOF-Net 玉米缺氮检测实验报告

本 Notebook 汇总三个 baseline（EfficientNet-B0 / ResNet18 / DeiT-Tiny）和当前 RMOF-Net 优化方案。

内容包括：数据检查、baseline 训练与最终测试、RMOF-Net 当前结果、消融实验、训练曲线和混淆矩阵。当前 notebook 末尾的“当前最终结果汇总”和“当前消融结果”已经直接写入现有 artifact，不需要重新训练即可查看。


In [ ]:
from __future__ import annotations

import copy
import json
import math
import random
import time
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

# ===== 只需要检查这里的数据路径；其余参数已经固定 =====
PROJECT_ROOT_OVERRIDE = None
CSV_PATH_OVERRIDE = None

# 固定配置：不做超参数搜索，三个 baseline 完全共享。
# DeiT-Tiny 的 torchvision 实现要求 224x224 输入，因此三组 baseline 统一用 224x224。
IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
BATCH_SIZE = 16
NUM_WORKERS = 0
EPOCHS = 25
FREEZE_EPOCHS = 4
PATIENCE = 6
FROZEN_HEAD_LR = 1e-3
FINETUNE_HEAD_LR = 3e-4
BACKBONE_LR = 3e-5
MIN_LR_RATIO = 0.05
WEIGHT_DECAY = 1e-2
LABEL_SMOOTHING = 0.05
CORN_WEIGHT = 0.30
GRAD_CLIP_NORM = 1.0
TOKEN_DIM = 128
ATTENTION_HEADS = 4
DROPOUT = 0.30
SEEDS = [42, 52, 62]
NUM_CLASSES = 3
CLASS_NAMES = ["N0", "N75", "NFull"]
PRETRAINED = True

# 保持 True 后只需 Run All 一次。已完成实验会自动跳过。
RUN_PIPELINE = True
RESUME_COMPLETED = True

# 固定版本目录，避免混入旧 LG-MAP / quick-run 结果。
RUN_NAME = "three_baselines_fixed_v1"
OUTPUT_DIR = Path("three_baselines_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not 0 < FREEZE_EPOCHS < EPOCHS:
    raise ValueError("FREEZE_EPOCHS 必须在 1 和 EPOCHS-1 之间")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print(f"Fixed run: {len(SEEDS)} seeds, max {EPOCHS} epochs, resume={RESUME_COMPLETED}")

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def locate_project() -> Tuple[Optional[Path], Optional[Path]]:
    if PROJECT_ROOT_OVERRIDE is not None:
        root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        csv_path = (
            Path(CSV_PATH_OVERRIDE).expanduser().resolve()
            if CSV_PATH_OVERRIDE is not None
            else root / "split_40_10_50.csv"
        )
        return root, csv_path

    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "EfficientN", cwd.parent / "EfficientN", cwd.parent]
    for root in candidates:
        for csv_path in (
            root / "split_40_10_50.csv",
            root / "data" / "split.csv",
        ):
            if csv_path.is_file():
                return root, csv_path
    return None, None


PROJECT_ROOT, CSV_PATH = locate_project()
DATA_AVAILABLE = PROJECT_ROOT is not None and CSV_PATH is not None
if DATA_AVAILABLE:
    print("Project root:", PROJECT_ROOT)
    print("CSV:", CSV_PATH)
else:
    print(
        "未找到数据。模型 smoke test 仍可运行。\n"
        "正式训练前请设置 PROJECT_ROOT_OVERRIDE 和 CSV_PATH_OVERRIDE。"
    )
set_seed(SEEDS[0])



## 1. 数据检查

Notebook 优先读取仓库现有的 split_40_10_50.csv。
三个 baseline 共享同一份 CSV。开发期间只使用训练集和验证集。



In [ ]:
REQUIRED_COLUMNS = {"filepath", "filename", "label_index", "split"}
split_frame: Optional[pd.DataFrame] = None


def audit_split(csv_path: Path, data_root: Path) -> pd.DataFrame:
    frame = pd.read_csv(csv_path)
    missing = REQUIRED_COLUMNS.difference(frame.columns)
    if missing:
        raise ValueError(f"CSV缺少列：{sorted(missing)}")

    frame = frame.copy()
    frame["label_index"] = frame["label_index"].astype(int)
    if not frame["label_index"].isin(range(NUM_CLASSES)).all():
        raise ValueError("label_index 必须为 0、1、2")
    duplicated = int(frame["filepath"].duplicated().sum())
    if duplicated:
        raise ValueError(f"CSV存在 {duplicated} 个重复 filepath")

    missing_files = []
    for value in frame["filepath"].astype(str):
        path = Path(value)
        resolved = path if path.is_absolute() else data_root / path
        if not resolved.is_file():
            missing_files.append(str(resolved))
    if missing_files:
        preview = "\n".join(missing_files[:5])
        raise FileNotFoundError(f"找不到 {len(missing_files)} 张图像，例如：\n{preview}")

    table = pd.crosstab(frame["split"], frame["label_index"])
    table = table.rename(columns=dict(enumerate(CLASS_NAMES)))
    display(table)

    split_sets = {
        name: set(group["filepath"].astype(str))
        for name, group in frame.groupby("split")
    }
    names = sorted(split_sets)
    for index, left in enumerate(names):
        for right in names[index + 1 :]:
            overlap = split_sets[left].intersection(split_sets[right])
            if overlap:
                raise ValueError(f"{left} 与 {right} 有 {len(overlap)} 个重复文件")
    print("文件级 split overlap：0")
    return frame


if DATA_AVAILABLE:
    split_frame = audit_split(CSV_PATH, PROJECT_ROOT)




In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class AspectResizePad:
    """保持比例缩放，再填充到固定尺寸。"""

    def __init__(
        self,
        height: int,
        width: int,
        fill: Tuple[int, int, int] = (124, 116, 104),
    ) -> None:
        self.height = height
        self.width = width
        self.fill = fill

    def __call__(self, image: Image.Image) -> Image.Image:
        width, height = image.size
        scale = min(self.width / width, self.height / height)
        new_width = max(1, round(width * scale))
        new_height = max(1, round(height * scale))
        resized = image.resize((new_width, new_height), Image.Resampling.BILINEAR)
        canvas = Image.new("RGB", (self.width, self.height), self.fill)
        canvas.paste(
            resized,
            ((self.width - new_width) // 2, (self.height - new_height) // 2),
        )
        return canvas


def build_transform(train: bool) -> transforms.Compose:
    steps: List[object] = [AspectResizePad(IMAGE_HEIGHT, IMAGE_WIDTH)]
    if train:
        steps.extend(
            [
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(8),
                transforms.RandomAffine(
                    degrees=0,
                    translate=(0.05, 0.05),
                    scale=(0.9, 1.1),
                ),
            ]
        )
    steps.extend(
        [
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]
    )
    return transforms.Compose(steps)


class MaizeNitrogenDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        data_root: Path,
        split: str,
        transform: transforms.Compose,
    ) -> None:
        self.frame = frame.loc[frame["split"] == split].reset_index(drop=True)
        if self.frame.empty:
            raise ValueError(f"split={split!r} 没有样本")
        self.data_root = data_root
        self.transform = transform

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        path = Path(str(row["filepath"]))
        path = path if path.is_absolute() else self.data_root / path
        with Image.open(path) as opened:
            image = opened.convert("RGB")
        return self.transform(image), int(row["label_index"]), str(row["filename"])


def make_loaders(seed: int, include_test: bool = False):
    """验证阶段不创建 test loader，最终测试阶段才允许读取测试集。"""
    if not DATA_AVAILABLE or split_frame is None:
        raise RuntimeError("未找到数据，不能建立 DataLoader")
    available = set(split_frame["split"].astype(str))
    val_name = "val" if "val" in available else "validation"
    if "train" not in available or val_name not in available:
        raise ValueError("CSV需要 train 和 val（或 validation）split")
    if include_test and "test" not in available:
        raise ValueError("最终测试需要 test split")

    train_dataset = MaizeNitrogenDataset(
        split_frame, PROJECT_ROOT, "train", build_transform(True)
    )
    val_dataset = MaizeNitrogenDataset(
        split_frame, PROJECT_ROOT, val_name, build_transform(False)
    )
    common = {
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pin_memory": DEVICE.type == "cuda",
    }
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset, shuffle=True, generator=generator, **common
    )
    val_loader = DataLoader(val_dataset, shuffle=False, **common)

    test_loader = None
    if include_test:
        test_dataset = MaizeNitrogenDataset(
            split_frame, PROJECT_ROOT, "test", build_transform(False)
        )
        test_loader = DataLoader(test_dataset, shuffle=False, **common)
    return train_loader, val_loader, test_loader


def denormalize(images: torch.Tensor) -> torch.Tensor:
    mean = images.new_tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std = images.new_tensor(IMAGENET_STD).view(1, 3, 1, 1)
    return (images * std + mean).clamp(0, 1)


if DATA_AVAILABLE:
    preview_train, _, _ = make_loaders(SEEDS[0])
    images, labels, names = next(iter(preview_train))
    print("Batch image shape:", tuple(images.shape))
    print("Label range:", int(labels.min()), int(labels.max()))
    shown = denormalize(images[:6]).permute(0, 2, 3, 1).numpy()
    fig, axes = plt.subplots(2, 3, figsize=(11, 6))
    for ax, image, label, name in zip(axes.flat, shown, labels, names):
        ax.imshow(image)
        ax.set_title(f"{name}\n{CLASS_NAMES[int(label)]}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 2. Baseline 模型

本次更正后只运行三个 baseline：

| 名称 | 内容 |
|---|---|
| B0_Base | EfficientNet-B0 baseline |
| ResNet18_Base | ResNet-18 baseline |
| DeiT_Tiny_Base | DeiT-Tiny Patch16 baseline |

下一个代码单元保留原 LG-MAP 辅助模块定义；随后会覆盖模型构建函数，只执行上述三个 baseline。



In [ ]:
def rgb_saturation(rgb: torch.Tensor) -> torch.Tensor:
    maximum = rgb.max(dim=1, keepdim=True).values
    minimum = rgb.min(dim=1, keepdim=True).values
    return (maximum - minimum) / maximum.clamp_min(1e-6)


def rgb_to_lab_b(rgb: torch.Tensor) -> torch.Tensor:
    linear = torch.where(
        rgb > 0.04045,
        ((rgb + 0.055) / 1.055).pow(2.4),
        rgb / 12.92,
    )
    matrix = rgb.new_tensor(
        [
            [0.4124564, 0.3575761, 0.1804375],
            [0.2126729, 0.7151522, 0.0721750],
            [0.0193339, 0.1191920, 0.9503041],
        ]
    )
    xyz = torch.einsum("ij,bjhw->bihw", matrix, linear)
    white = rgb.new_tensor([0.95047, 1.0, 1.08883]).view(1, 3, 1, 1)
    xyz = xyz / white
    delta = 6.0 / 29.0
    f_xyz = torch.where(
        xyz > delta**3,
        xyz.clamp_min(0).pow(1.0 / 3.0),
        xyz / (3.0 * delta**2) + 4.0 / 29.0,
    )
    return 200.0 * (f_xyz[:, 1:2] - f_xyz[:, 2:3]) / 128.0


class SoftColourPrior(nn.Module):
    """ExG、Saturation、Lab b* 生成软先验，不执行硬分割。"""

    def __init__(self) -> None:
        super().__init__()
        self.threshold = nn.Parameter(torch.tensor(0.0))
        self.log_sharpness = nn.Parameter(torch.tensor(math.log(math.expm1(2.0))))

    def forward(self, rgb: torch.Tensor) -> torch.Tensor:
        r, g, b = rgb[:, 0:1], rgb[:, 1:2], rgb[:, 2:3]
        cues = torch.cat(
            (2.0 * g - r - b, rgb_saturation(rgb), rgb_to_lab_b(rgb)),
            dim=1,
        )
        mean = cues.mean(dim=(2, 3), keepdim=True)
        std = cues.std(dim=(2, 3), keepdim=True, unbiased=False).clamp_min(1e-5)
        standardized = (cues - mean) / std
        combined = torch.logsumexp(standardized, dim=1, keepdim=True)
        combined = combined - math.log(3.0)
        sharpness = F.softplus(self.log_sharpness)
        return torch.sigmoid(sharpness * (combined - self.threshold))


class FixedMultiScale2x2(nn.Module):
    def __init__(self, token_dim: int = 128) -> None:
        super().__init__()
        self.project3 = nn.Sequential(
            nn.Linear(112, token_dim), nn.GELU(), nn.LayerNorm(token_dim)
        )
        self.project4 = nn.Sequential(
            nn.Linear(320, token_dim), nn.GELU(), nn.LayerNorm(token_dim)
        )

    @staticmethod
    def _pool(feature: torch.Tensor) -> torch.Tensor:
        global_token = F.adaptive_avg_pool2d(feature, 1).flatten(2)
        regions = F.adaptive_avg_pool2d(feature, 2).flatten(2)
        return torch.cat((global_token, regions), dim=2).transpose(1, 2)

    def forward(self, stage3: torch.Tensor, stage4: torch.Tensor) -> torch.Tensor:
        tokens3 = self.project3(self._pool(stage3))
        tokens4 = self.project4(self._pool(stage4))
        return torch.cat((tokens3, tokens4), dim=1).mean(dim=1)


class LGMAP(nn.Module):
    """Leaf-Guided Multi-scale Attentive Pooling."""

    def __init__(
        self,
        token_dim: int = 128,
        attention_heads: int = 4,
        use_colour_prior: bool = False,
    ) -> None:
        super().__init__()
        self.project3 = nn.Conv2d(112, token_dim, 1)
        self.project4 = nn.Conv2d(320, token_dim, 1)
        self.scale_logits = nn.Parameter(torch.zeros(2))
        self.refine = nn.Sequential(
            nn.Conv2d(token_dim, token_dim, 3, padding=1, groups=token_dim),
            nn.GroupNorm(8, token_dim),
            nn.GELU(),
            nn.Conv2d(token_dim, token_dim, 1),
            nn.GroupNorm(8, token_dim),
            nn.GELU(),
        )
        self.attention_score = nn.Conv2d(token_dim, attention_heads, 1)
        self.local_merge = nn.Linear(attention_heads * token_dim, token_dim)
        self.colour_prior = SoftColourPrior() if use_colour_prior else None
        self.beta_raw = nn.Parameter(torch.tensor(0.0))
        nn.init.zeros_(self.attention_score.weight)
        nn.init.zeros_(self.attention_score.bias)

    def forward(
        self,
        stage3: torch.Tensor,
        stage4: torch.Tensor,
        rgb: torch.Tensor,
    ):
        f3 = self.project3(stage3)
        f4 = F.interpolate(
            self.project4(stage4),
            size=f3.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        weights = self.scale_logits.softmax(dim=0)
        fused = weights[0] * f3 + weights[1] * f4
        fused = fused + self.refine(fused)

        score = self.attention_score(fused).flatten(2)
        mask = None
        if self.colour_prior is not None:
            mask = self.colour_prior(rgb)
            resized = F.interpolate(
                mask,
                size=fused.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
            beta = 2.0 * torch.tanh(self.beta_raw)
            score = score + beta * resized.flatten(2).clamp_min(1e-4).log()

        attention = score.softmax(dim=-1)
        flat_feature = fused.flatten(2)
        local_tokens = torch.einsum("bhp,bdp->bhd", attention, flat_feature)
        local = self.local_merge(local_tokens.flatten(1))
        global_feature = F.adaptive_avg_pool2d(fused, 1).flatten(1)
        return torch.cat((global_feature, local), dim=1), attention, mask




In [ ]:
VARIANT_CONFIGS = {
    "B0_Base": {"kind": "base", "colour": False, "corn": False},
    "B0_MS2x2": {"kind": "ms2x2", "colour": False, "corn": False},
    "B0_LGMAP": {"kind": "lgmap", "colour": False, "corn": False},
    "B0_LGMAP_Color": {"kind": "lgmap", "colour": True, "corn": False},
    "B0_LGMAP_Color_CORN": {"kind": "lgmap", "colour": True, "corn": True},
}


class EfficientNetNitrogenModel(nn.Module):
    def __init__(
        self,
        variant: str,
        pretrained: bool = True,
        token_dim: int = 128,
        attention_heads: int = 4,
        dropout: float = 0.30,
    ) -> None:
        super().__init__()
        if variant not in VARIANT_CONFIGS:
            raise ValueError(f"Unknown variant: {variant}")
        self.variant = variant
        self.config = VARIANT_CONFIGS[variant]

        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        backbone = models.efficientnet_b0(weights=weights)
        self.features = backbone.features
        self.avgpool = backbone.avgpool
        self.base_classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(1280, NUM_CLASSES)
        )

        self.fixed_pool = None
        self.lgmap = None
        self.aux_classifier = None
        self.corn_head = None

        if self.config["kind"] == "ms2x2":
            self.fixed_pool = FixedMultiScale2x2(token_dim)
            self.aux_classifier = nn.Sequential(
                nn.LayerNorm(token_dim),
                nn.Dropout(dropout),
                nn.Linear(token_dim, NUM_CLASSES),
            )
        elif self.config["kind"] == "lgmap":
            self.lgmap = LGMAP(
                token_dim, attention_heads, self.config["colour"]
            )
            summary_dim = 2 * token_dim
            self.aux_classifier = nn.Sequential(
                nn.LayerNorm(summary_dim),
                nn.Dropout(dropout),
                nn.Linear(summary_dim, NUM_CLASSES),
            )
            if self.config["corn"]:
                self.corn_head = nn.Sequential(
                    nn.LayerNorm(summary_dim), nn.Linear(summary_dim, 2)
                )

        if self.aux_classifier is not None:
            nn.init.zeros_(self.aux_classifier[-1].weight)
            nn.init.zeros_(self.aux_classifier[-1].bias)

        self.register_buffer(
            "image_mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
        )
        self.register_buffer(
            "image_std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
        )

    def extract_features(self, images: torch.Tensor):
        stage3 = stage4 = None
        x = images
        for index, block in enumerate(self.features):
            x = block(x)
            if index == 5:
                stage3 = x
            elif index == 7:
                stage4 = x
        if stage3 is None or stage4 is None:
            raise RuntimeError("EfficientNet中间特征提取失败")
        return stage3, stage4, x

    def forward(self, images: torch.Tensor) -> Dict[str, Optional[torch.Tensor]]:
        stage3, stage4, final_feature = self.extract_features(images)
        base_feature = self.avgpool(final_feature).flatten(1)
        base_logits = self.base_classifier(base_feature)

        if self.config["kind"] == "base":
            return {
                "logits": base_logits,
                "base_logits": base_logits,
                "aux_logits": None,
                "corn_logits": None,
                "attention": None,
                "attention_size": None,
                "mask": None,
            }

        if self.config["kind"] == "ms2x2":
            summary = self.fixed_pool(stage3, stage4)
            attention = mask = None
            attention_size = None
        else:
            rgb = (images * self.image_std + self.image_mean).clamp(0, 1)
            summary, attention, mask = self.lgmap(stage3, stage4, rgb)
            attention_size = stage3.shape[-2:]

        aux_logits = self.aux_classifier(summary)
        corn_logits = self.corn_head(summary) if self.corn_head is not None else None
        return {
            "logits": base_logits + aux_logits,
            "base_logits": base_logits,
            "aux_logits": aux_logits,
            "corn_logits": corn_logits,
            "attention": attention,
            "attention_size": attention_size,
            "mask": mask,
        }


def build_model(variant: str, pretrained: bool = PRETRAINED):
    return EfficientNetNitrogenModel(
        variant, pretrained, TOKEN_DIM, ATTENTION_HEADS, DROPOUT
    )


def parameter_count(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())



In [ ]:
# ===== 更正：仅运行三个 baseline（EfficientNet-B0、ResNet-18、DeiT-Tiny） =====
VARIANT_CONFIGS = {
    "B0_Base": {"arch": "efficientnet_b0", "corn": False},
    "ResNet18_Base": {"arch": "resnet18", "corn": False},
    "DeiT_Tiny_Base": {"arch": "deit_tiny", "corn": False},
}


def torchvision_weights(name: str, pretrained: bool):
    if not pretrained:
        return None
    weights_cls = getattr(models, name, None)
    if weights_cls is None:
        return None
    return weights_cls.DEFAULT


def unique_parameters(parameters):
    seen = set()
    unique = []
    for parameter in parameters:
        if id(parameter) in seen:
            continue
        seen.add(id(parameter))
        unique.append(parameter)
    return unique


def standard_output(logits: torch.Tensor) -> Dict[str, Optional[torch.Tensor]]:
    return {
        "logits": logits,
        "base_logits": logits,
        "aux_logits": None,
        "corn_logits": None,
        "attention": None,
        "attention_size": None,
        "mask": None,
    }


def make_deit_tiny(pretrained: bool) -> Tuple[nn.Module, str]:
    if hasattr(models, "deit_tiny_patch16_224"):
        weights = torchvision_weights("DeiT_Tiny_Patch16_224_Weights", pretrained)
        model = models.deit_tiny_patch16_224(weights=weights)
        in_features = model.heads.head.in_features
        model.heads.head = nn.Linear(in_features, NUM_CLASSES)
        return model, "torchvision"

    try:
        import timm
    except ImportError as exc:
        raise RuntimeError(
            "当前 torchvision 没有 deit_tiny_patch16_224；请安装 timm 或升级 torchvision。"
        ) from exc
    model = timm.create_model(
        "deit_tiny_patch16_224",
        pretrained=pretrained,
        num_classes=NUM_CLASSES,
        img_size=IMAGE_HEIGHT,
    )
    return model, "timm"


class BaselineNitrogenModel(nn.Module):
    def __init__(self, variant: str, pretrained: bool = True, dropout: float = 0.30) -> None:
        super().__init__()
        if variant not in VARIANT_CONFIGS:
            raise ValueError(f"Unknown variant: {variant}")
        self.variant = variant
        self.config = VARIANT_CONFIGS[variant]
        self.arch = self.config["arch"]
        self.deit_backend = None

        if self.arch == "efficientnet_b0":
            weights = torchvision_weights("EfficientNet_B0_Weights", pretrained)
            backbone = models.efficientnet_b0(weights=weights)
            self.features = backbone.features
            self.pool = backbone.avgpool
            self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(1280, NUM_CLASSES))
        elif self.arch == "resnet18":
            weights = torchvision_weights("ResNet18_Weights", pretrained)
            backbone = models.resnet18(weights=weights)
            self.features = nn.Sequential(
                backbone.conv1,
                backbone.bn1,
                backbone.relu,
                backbone.maxpool,
                backbone.layer1,
                backbone.layer2,
                backbone.layer3,
                backbone.layer4,
            )
            self.pool = backbone.avgpool
            self.classifier = nn.Sequential(
                nn.Dropout(dropout), nn.Linear(backbone.fc.in_features, NUM_CLASSES)
            )
        elif self.arch == "deit_tiny":
            self.deit, self.deit_backend = make_deit_tiny(pretrained)
            self.features = self._deit_feature_blocks()
        else:
            raise ValueError(f"Unsupported architecture: {self.arch}")

    def _deit_feature_blocks(self) -> nn.ModuleList:
        if self.deit_backend == "torchvision":
            return nn.ModuleList(
                [self.deit.conv_proj, *list(self.deit.encoder.layers), self.deit.encoder.ln]
            )
        return nn.ModuleList([self.deit.patch_embed, *list(self.deit.blocks), self.deit.norm])

    def backbone_parameters(self):
        if self.arch != "deit_tiny":
            return unique_parameters(self.features.parameters())
        head = self.deit.heads if self.deit_backend == "torchvision" else self.deit.head
        head_ids = {id(parameter) for parameter in head.parameters()}
        return unique_parameters(
            parameter for parameter in self.deit.parameters() if id(parameter) not in head_ids
        )

    def late_backbone_parameters(self):
        if self.arch != "deit_tiny":
            return unique_parameters(self.features[5:].parameters())
        if self.deit_backend == "torchvision":
            parameters = [self.deit.class_token]
            parameters.extend(self.deit.encoder.layers[-4:].parameters())
            parameters.extend(self.deit.encoder.ln.parameters())
        else:
            parameters = [self.deit.cls_token, self.deit.pos_embed]
            parameters.extend(self.deit.blocks[-4:].parameters())
            parameters.extend(self.deit.norm.parameters())
        return unique_parameters(parameters)

    def head_parameters(self):
        backbone_ids = {id(parameter) for parameter in self.backbone_parameters()}
        return unique_parameters(
            parameter for parameter in self.parameters() if id(parameter) not in backbone_ids
        )

    def set_frozen_backbone_eval(self, backbone_unfrozen: bool) -> None:
        blocks = list(self.features)
        frozen_blocks = blocks[:5] if backbone_unfrozen else blocks
        for block in frozen_blocks:
            block.eval()

    def forward(self, images: torch.Tensor) -> Dict[str, Optional[torch.Tensor]]:
        if self.arch == "deit_tiny":
            logits = self.deit(images)
        else:
            feature = self.features(images)
            pooled = self.pool(feature).flatten(1)
            logits = self.classifier(pooled)
        return standard_output(logits)


def build_model(variant: str, pretrained: bool = PRETRAINED):
    return BaselineNitrogenModel(variant, pretrained, DROPOUT)


def parameter_count(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters())


print("Baseline variants:", ", ".join(VARIANT_CONFIGS))


## 3. 损失与指标

结构比较统一使用 Cross-Entropy。完整模型使用：

    total_loss = ce_loss + 0.3 * corn_loss

CORN 的两个条件任务为 y > 0 与 y > 1。



In [ ]:
cross_entropy = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)


def corn_loss(corn_logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    first = F.binary_cross_entropy_with_logits(
        corn_logits[:, 0], (target > 0).float()
    )
    second_mask = target > 0
    if second_mask.any():
        second = F.binary_cross_entropy_with_logits(
            corn_logits[second_mask, 1],
            (target[second_mask] > 1).float(),
        )
        return 0.5 * (first + second)
    return first


def objective(output, target: torch.Tensor, use_corn: bool):
    ce = cross_entropy(output["logits"], target)
    corn = ce.new_zeros(())
    if use_corn:
        if output["corn_logits"] is None:
            raise ValueError("CORN实验缺少 corn_logits")
        corn = corn_loss(output["corn_logits"], target)
    return {"total": ce + CORN_WEIGHT * corn, "ce": ce, "corn": corn}


def classification_metrics(labels, predictions) -> Dict[str, float]:
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, labels=[0, 1, 2], zero_division=0
    )
    deficient_true = labels != 2
    deficient_pred = predictions != 2
    endpoint = int(
        (
            ((labels == 0) & (predictions == 2))
            | ((labels == 2) & (predictions == 0))
        ).sum()
    )
    metrics = {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(f1.mean()),
        "n0_f1": float(f1[0]),
        "n75_f1": float(f1[1]),
        "nfull_f1": float(f1[2]),
        "deficiency_f1": float(
            f1_score(deficient_true, deficient_pred, zero_division=0)
        ),
        "ordinal_mae": float(np.abs(labels - predictions).mean()),
        "endpoint_reversals": endpoint,
    }
    for index, name in enumerate(("n0", "n75", "nfull")):
        metrics[f"{name}_precision"] = float(precision[index])
        metrics[f"{name}_recall"] = float(recall[index])
    return metrics


def plot_confusion(labels, predictions, title: str, ax=None):
    matrix = confusion_matrix(labels, predictions, labels=[0, 1, 2])
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 3.5))
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    return matrix



## 4. 模型 smoke test

不需要真实数据。检查三个 baseline 的前向、反向和参数量。



In [ ]:
def run_model_smoke_test() -> None:
    set_seed(42)
    synthetic = torch.randn(2, 3, IMAGE_HEIGHT, IMAGE_WIDTH, device=DEVICE)
    target = torch.tensor([0, 2], device=DEVICE)
    counts = {}

    for variant in VARIANT_CONFIGS:
        model = build_model(variant, pretrained=False).to(DEVICE)
        model.train()
        optimizer = torch.optim.Adam(model.head_parameters(), lr=1e-3)
        optimizer.zero_grad(set_to_none=True)
        output = model(synthetic)
        assert output["logits"].shape == (2, NUM_CLASSES)
        assert torch.isfinite(output["logits"]).all()
        loss = cross_entropy(output["logits"], target)
        loss.backward()
        head_grad = sum(
            float(parameter.grad.detach().abs().sum())
            for parameter in model.head_parameters()
            if parameter.grad is not None
        )
        assert head_grad > 0, variant
        optimizer.step()
        counts[variant] = parameter_count(model)
        del model, optimizer

    print("Smoke test passed.")
    for name, count in counts.items():
        print(f"{name:25s}: {count / 1e6:.3f} M parameters")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


run_model_smoke_test()



## 5. 固定训练策略

- 前 4 个 epoch 冻结主干，以 `1e-3` 训练新分类头和新模块；
- 之后只解冻各 backbone 的后段模块，分类头使用 `3e-4`、主干使用 `3e-5`；
- 微调阶段使用 cosine decay，最低保留 5% 学习率；
- BatchNorm running statistics 始终冻结，梯度范数裁剪为 1.0；
- 最多 25 个 epoch，解冻后按验证集 Macro-F1 连续 6 个 epoch 无提升才早停。

这些参数对三个 baseline 完全一致，不在测试集上调参。

In [ ]:
def keep_batchnorm_eval(model: nn.Module) -> None:
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()


def cosine_lr(start_lr: float, progress: float) -> float:
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return start_lr * (MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * cosine)


def state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
    use_corn,
    backbone_unfrozen,
):
    model.train()
    if hasattr(model, "set_frozen_backbone_eval"):
        model.set_frozen_backbone_eval(backbone_unfrozen)
    else:
        for block in model.features[:5]:
            block.eval()
        if not backbone_unfrozen:
            for block in model.features[5:]:
                block.eval()
    keep_batchnorm_eval(model)
    labels, predictions = [], []
    total_loss = 0.0
    total_samples = 0

    for images, target, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
            output = model(images)
            losses = objective(output, target, use_corn)
        scaler.scale(losses["total"]).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        batch_size = target.shape[0]
        total_loss += losses["total"].item() * batch_size
        total_samples += batch_size
        labels.extend(target.detach().cpu().tolist())
        predictions.extend(output["logits"].argmax(1).detach().cpu().tolist())

    metrics = classification_metrics(labels, predictions)
    metrics["loss"] = total_loss / total_samples
    return metrics


@torch.inference_mode()
def evaluate_model(model, loader, use_corn):
    model.eval()
    labels, predictions, filenames = [], [], []
    total_loss = 0.0
    total_samples = 0
    elapsed = 0.0

    for images, target, names in loader:
        images = images.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
            output = model(images)
            losses = objective(output, target, use_corn)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        elapsed += time.perf_counter() - start

        prediction = output["logits"].argmax(1)
        batch_size = target.shape[0]
        total_loss += losses["total"].item() * batch_size
        total_samples += batch_size
        labels.extend(target.cpu().tolist())
        predictions.extend(prediction.cpu().tolist())
        filenames.extend(list(names))

    metrics = classification_metrics(labels, predictions)
    metrics["loss"] = total_loss / total_samples
    metrics["inference_ms_per_image"] = 1000.0 * elapsed / total_samples
    return metrics, labels, predictions, filenames


def fit_model(
    variant: str,
    seed: int,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = EPOCHS,
    freeze_epochs: int = FREEZE_EPOCHS,
):
    set_seed(seed)
    model = build_model(variant, pretrained=PRETRAINED).to(DEVICE)
    use_corn = VARIANT_CONFIGS[variant]["corn"]

    if hasattr(model, "backbone_parameters"):
        all_backbone = list(model.backbone_parameters())
        late_backbone = list(model.late_backbone_parameters())
        head_parameters = list(model.head_parameters())
    else:
        all_backbone = list(model.features.parameters())
        late_backbone = list(model.features[5:].parameters())
        backbone_ids = {id(parameter) for parameter in all_backbone}
        head_parameters = [
            parameter for parameter in model.parameters()
            if id(parameter) not in backbone_ids
        ]
    for parameter in all_backbone:
        parameter.requires_grad_(False)

    optimizer = torch.optim.AdamW(
        [
            {"params": late_backbone, "lr": 0.0, "name": "backbone"},
            {"params": head_parameters, "lr": FROZEN_HEAD_LR, "name": "head"},
        ],
        weight_decay=WEIGHT_DECAY,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
    best_score = (-float("inf"), -float("inf"))
    best_state = state_dict_to_cpu(model)
    best_epoch = 0
    stale = 0
    history = []

    for epoch in range(epochs):
        if epoch == freeze_epochs:
            for parameter in late_backbone:
                parameter.requires_grad_(True)
            stale = 0

        if epoch < freeze_epochs:
            head_lr = FROZEN_HEAD_LR
            backbone_lr = 0.0
        else:
            progress = (epoch - freeze_epochs) / max(
                1, epochs - freeze_epochs - 1
            )
            head_lr = cosine_lr(FINETUNE_HEAD_LR, progress)
            backbone_lr = cosine_lr(BACKBONE_LR, progress)
        optimizer.param_groups[0]["lr"] = backbone_lr
        optimizer.param_groups[1]["lr"] = head_lr

        train_metrics = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scaler,
            use_corn,
            backbone_unfrozen=epoch >= freeze_epochs,
        )
        val_metrics, _, _, _ = evaluate_model(model, val_loader, use_corn)
        history.append(
            {
                "epoch": epoch + 1,
                "train_loss": train_metrics["loss"],
                "train_macro_f1": train_metrics["macro_f1"],
                "val_loss": val_metrics["loss"],
                "val_macro_f1": val_metrics["macro_f1"],
                "val_n75_f1": val_metrics["n75_f1"],
                "head_lr": head_lr,
                "backbone_lr": backbone_lr,
            }
        )
        print(
            f"{variant} seed={seed} epoch={epoch + 1:02d} "
            f"trainF1={train_metrics['macro_f1']:.4f} "
            f"valF1={val_metrics['macro_f1']:.4f} "
            f"N75={val_metrics['n75_f1']:.4f}"
        )

        score = (val_metrics["macro_f1"], val_metrics["n75_f1"])
        if score > best_score:
            best_score = score
            best_state = state_dict_to_cpu(model)
            best_epoch = epoch + 1
            stale = 0
        elif epoch >= freeze_epochs:
            stale += 1
            if stale >= PATIENCE:
                print("Early stopping after backbone unfreezing.")
                break

    model.load_state_dict(best_state)
    val_metrics, labels, predictions, filenames = evaluate_model(
        model, val_loader, use_corn
    )
    run_dir = OUTPUT_DIR / variant / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / "best_model.pt"
    torch.save(
        {
            "variant": variant,
            "seed": seed,
            "best_epoch": best_epoch,
            "state_dict": best_state,
        },
        checkpoint_path,
    )
    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    pd.DataFrame(
        {"filename": filenames, "label": labels, "prediction": predictions}
    ).to_csv(run_dir / "validation_predictions.csv", index=False)

    val_metrics.update(
        {
            "variant": variant,
            "seed": int(seed),
            "best_epoch": int(best_epoch),
            "parameters": int(parameter_count(model)),
            "checkpoint": str(checkpoint_path.resolve()),
        }
    )
    with (run_dir / "validation_metrics.json").open("w", encoding="utf-8") as file:
        json.dump(val_metrics, file, ensure_ascii=False, indent=2)
    return model, pd.DataFrame(history), val_metrics

## 6. 一次运行三个 baseline

以下固定执行 `3 个 baseline × 3 个 seed`。这属于基线对比，不是超参数搜索。
验证阶段不会创建 test loader；全部验证完成后再统一进入最终测试。

每个完整 run 都保存 checkpoint、训练历史和验证指标。运行意外中断后再次 **Run All**，
程序会从第一个未完成的 run 继续。

In [ ]:
EXPERIMENT_ORDER = ["B0_Base", "ResNet18_Base", "DeiT_Tiny_Base"]


def fixed_run_manifest() -> Dict[str, object]:
    return {
        "run_name": RUN_NAME,
        "csv_path": str(CSV_PATH.resolve()) if CSV_PATH is not None else None,
        "image_size": [IMAGE_HEIGHT, IMAGE_WIDTH],
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "freeze_epochs": FREEZE_EPOCHS,
        "patience": PATIENCE,
        "frozen_head_lr": FROZEN_HEAD_LR,
        "finetune_head_lr": FINETUNE_HEAD_LR,
        "backbone_lr": BACKBONE_LR,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "corn_weight": CORN_WEIGHT,
        "token_dim": TOKEN_DIM,
        "attention_heads": ATTENTION_HEADS,
        "dropout": DROPOUT,
        "seeds": SEEDS,
        "variants": EXPERIMENT_ORDER,
    }


def verify_or_create_manifest() -> None:
    path = OUTPUT_DIR / "fixed_run_manifest.json"
    current = fixed_run_manifest()
    if path.is_file():
        saved = json.loads(path.read_text(encoding="utf-8"))
        if saved != current:
            raise RuntimeError(
                "输出目录中存在另一套参数或数据的结果。"
                "为防止混合实验，请恢复固定配置或更换 OUTPUT_DIR。"
            )
    else:
        path.write_text(
            json.dumps(current, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )


def load_completed_validation(variant: str, seed: int):
    run_dir = OUTPUT_DIR / variant / f"seed_{seed}"
    metrics_path = run_dir / "validation_metrics.json"
    checkpoint_path = run_dir / "best_model.pt"
    if not (RESUME_COMPLETED and metrics_path.is_file() and checkpoint_path.is_file()):
        return None
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    if metrics.get("variant") != variant or int(metrics.get("seed", -1)) != seed:
        return None
    metrics["checkpoint"] = str(checkpoint_path.resolve())
    print(f"[resume] 跳过已完成实验：{variant}, seed={seed}")
    return metrics


def run_ablation_suite():
    if not DATA_AVAILABLE:
        raise RuntimeError("请先配置数据路径")
    verify_or_create_manifest()
    total = len(EXPERIMENT_ORDER) * len(SEEDS)
    print(f"开始固定 baseline：{total} runs；无需手动选择 quick/full。")
    rows = []

    for variant in EXPERIMENT_ORDER:
        for seed in SEEDS:
            completed = load_completed_validation(variant, seed)
            if completed is not None:
                rows.append(completed)
                continue

            train_loader, val_loader, _ = make_loaders(seed, include_test=False)
            model, _, metrics = fit_model(
                variant,
                seed,
                train_loader,
                val_loader,
            )
            rows.append(metrics)
            del model, train_loader, val_loader
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    results = pd.DataFrame(rows)
    results.to_csv(OUTPUT_DIR / "ablation_results.csv", index=False)
    return results


ablation_results = None
if RUN_PIPELINE and DATA_AVAILABLE:
    ablation_results = run_ablation_suite()
    display(ablation_results)
elif (OUTPUT_DIR / "ablation_results.csv").is_file():
    ablation_results = pd.read_csv(OUTPUT_DIR / "ablation_results.csv")
    display(ablation_results)
elif RUN_PIPELINE:
    print("未找到数据：设置 PROJECT_ROOT_OVERRIDE / CSV_PATH_OVERRIDE 后 Run All。")
else:
    print("RUN_PIPELINE=False，训练未启动。")

In [ ]:
SUMMARY_METRICS = [
    "accuracy",
    "macro_f1",
    "n0_f1",
    "n75_f1",
    "nfull_f1",
    "deficiency_f1",
    "ordinal_mae",
    "endpoint_reversals",
    "parameters",
    "inference_ms_per_image",
]


def summarize_ablations(results: pd.DataFrame) -> pd.DataFrame:
    available = [column for column in SUMMARY_METRICS if column in results]
    mean = results.groupby("variant")[available].mean()
    std = results.groupby("variant")[available].std().fillna(0.0)
    summary = pd.DataFrame(index=mean.index)
    for metric in available:
        summary[f"{metric}_mean"] = mean[metric]
        summary[f"{metric}_std"] = std[metric]
    order = [name for name in EXPERIMENT_ORDER if name in summary.index]
    return summary.loc[order].reset_index()


def select_best_baseline(summary: pd.DataFrame) -> str:
    if summary.empty:
        raise RuntimeError("没有可选择的 baseline")
    return summary.sort_values(
        ["macro_f1_mean", "n75_f1_mean", "ordinal_mae_mean"],
        ascending=[False, False, True],
    ).iloc[0]["variant"]


ablation_summary = None
validation_best_variant = None
if ablation_results is not None and not ablation_results.empty:
    ablation_summary = summarize_ablations(ablation_results)
    columns = [
        "variant",
        "accuracy_mean",
        "macro_f1_mean",
        "macro_f1_std",
        "n75_f1_mean",
        "ordinal_mae_mean",
        "parameters_mean",
    ]
    display(ablation_summary[columns].round(4))
    ablation_summary.to_csv(OUTPUT_DIR / "ablation_summary.csv", index=False)

    if set(EXPERIMENT_ORDER).issubset(set(ablation_summary["variant"])):
        validation_best_variant = select_best_baseline(ablation_summary)
        base_f1 = float(
            ablation_summary.loc[
                ablation_summary["variant"] == "B0_Base", "macro_f1_mean"
            ].iloc[0]
        )
        best_f1 = float(
            ablation_summary.loc[
                ablation_summary["variant"] == validation_best_variant,
                "macro_f1_mean",
            ].iloc[0]
        )
        print(
            f"验证集最佳 baseline：{validation_best_variant}; "
            f"相对 B0_Base ΔMacro-F1={best_f1 - base_f1:+.4f}"
        )

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(
        ablation_summary["variant"],
        ablation_summary["macro_f1_mean"],
        yerr=ablation_summary["macro_f1_std"],
        capsize=4,
        color="#2F67B1",
    )
    ax.set_ylabel("Validation Macro-F1")
    ax.set_title("Three Baselines")
    ax.tick_params(axis="x", rotation=25)
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "validation_baselines.png",
        dpi=250,
        bbox_inches="tight",
    )
    plt.show()


def plot_saved_histories(seed: int = 42) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    found = False
    for variant in EXPERIMENT_ORDER:
        path = OUTPUT_DIR / variant / f"seed_{seed}" / "history.csv"
        if not path.is_file():
            continue
        found = True
        history = pd.read_csv(path)
        axes[0].plot(
            history["epoch"], history["val_macro_f1"], marker="o", label=variant
        )
        axes[1].plot(
            history["epoch"], history["val_n75_f1"], marker="o", label=variant
        )
    if not found:
        plt.close(fig)
        print("尚无训练曲线。")
        return
    axes[0].set_title("Validation Macro-F1")
    axes[1].set_title("Validation N75 F1")
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.set_ylabel("F1")
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / f"training_curves_seed_{seed}.png",
        dpi=250,
        bbox_inches="tight",
    )
    plt.show()


plot_saved_histories(SEEDS[0])

## 7. 三个 baseline 最终测试

验证集三 seed baseline 完整后，程序按平均 Macro-F1、N75 F1、Ordinal MAE 排序；
最终测试对三个 baseline 全部执行，测试集不会参与模型或超参数选择。

测试结果同样支持断点续跑。

In [ ]:
def load_trained_model(variant: str, checkpoint: str) -> nn.Module:
    model = build_model(variant, pretrained=False).to(DEVICE)
    saved = torch.load(checkpoint, map_location=DEVICE)
    model.load_state_dict(saved["state_dict"])
    model.eval()
    return model


def require_complete_ablation(results: pd.DataFrame) -> None:
    expected = {(variant, seed) for variant in EXPERIMENT_ORDER for seed in SEEDS}
    actual = {
        (str(row.variant), int(row.seed))
        for row in results[["variant", "seed"]].itertuples(index=False)
    }
    missing = sorted(expected.difference(actual))
    if missing:
        raise RuntimeError(f"baseline 尚未完整，缺少：{missing}")


def load_completed_test(variant: str, seed: int):
    run_dir = OUTPUT_DIR / variant / f"seed_{seed}"
    metrics_path = run_dir / "test_metrics.json"
    predictions_path = run_dir / "test_predictions.csv"
    if not (
        RESUME_COMPLETED
        and metrics_path.is_file()
        and predictions_path.is_file()
    ):
        return None
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    predictions = pd.read_csv(predictions_path)
    print(f"[resume] 跳过已完成测试：{variant}, seed={seed}")
    return metrics, (
        predictions["label"].astype(int).tolist(),
        predictions["prediction"].astype(int).tolist(),
        predictions["filename"].astype(str).tolist(),
    )


def save_decision_report(
    validation_results: pd.DataFrame,
    test_results: pd.DataFrame,
    best_variant: str,
) -> None:
    validation_summary = summarize_ablations(validation_results).set_index("variant")
    test_summary = summarize_ablations(test_results).set_index("variant")
    val_gain = (
        validation_summary.loc[best_variant, "macro_f1_mean"]
        - validation_summary.loc["B0_Base", "macro_f1_mean"]
    )
    test_gain = (
        test_summary.loc[best_variant, "macro_f1_mean"]
        - test_summary.loc["B0_Base", "macro_f1_mean"]
    )
    seed_pairs = validation_results.pivot(
        index="seed", columns="variant", values="macro_f1"
    )
    seed_wins = int((seed_pairs[best_variant] > seed_pairs["B0_Base"]).sum())
    if best_variant == "B0_Base":
        verdict = "B0_Base 是验证集平均 Macro-F1 最高的 baseline。"
    else:
        verdict = f"{best_variant} 是验证集平均 Macro-F1 最高的 baseline。"

    report = f"""# 三个 baseline 固定实验结论

- 验证集最佳 baseline：`{best_variant}`
- 验证集相对 B0_Base ΔMacro-F1：`{val_gain:+.4f}`
- 三个 seed 中优于 B0_Base：`{seed_wins}/3`
- 最终测试相对 B0_Base ΔMacro-F1：`{test_gain:+.4f}`
- 判断：{verdict}

三个 baseline 共享同一套固定参数；最终测试只在验证汇总完成后运行一次。
"""
    (OUTPUT_DIR / "decision_report.md").write_text(report, encoding="utf-8")
    print(report)


def run_final_test(results: pd.DataFrame):
    require_complete_ablation(results)
    summary = summarize_ablations(results)
    best_variant = summary.sort_values(
        ["macro_f1_mean", "n75_f1_mean", "ordinal_mae_mean"],
        ascending=[False, False, True],
    ).iloc[0]["variant"]
    selected = [variant for variant in EXPERIMENT_ORDER if variant in set(results["variant"])]
    rows = []
    prediction_store = {}

    for variant in selected:
        variant_rows = results.loc[results["variant"] == variant]
        for _, run in variant_rows.iterrows():
            seed = int(run["seed"])
            completed = load_completed_test(variant, seed)
            if completed is not None:
                metrics, stored_predictions = completed
                rows.append(metrics)
                prediction_store[(variant, seed)] = stored_predictions
                continue

            _, _, test_loader = make_loaders(seed, include_test=True)
            if test_loader is None:
                raise RuntimeError("test loader 创建失败")
            model = load_trained_model(variant, str(run["checkpoint"]))
            metrics, labels, predictions, filenames = evaluate_model(
                model, test_loader, VARIANT_CONFIGS[variant]["corn"]
            )
            metrics.update(
                {
                    "variant": variant,
                    "seed": seed,
                    "parameters": int(parameter_count(model)),
                }
            )
            rows.append(metrics)
            prediction_store[(variant, seed)] = (labels, predictions, filenames)
            run_dir = OUTPUT_DIR / variant / f"seed_{seed}"
            predictions_frame = pd.DataFrame(
                {
                    "filename": filenames,
                    "label": labels,
                    "prediction": predictions,
                }
            )
            predictions_frame.to_csv(run_dir / "test_predictions.csv", index=False)
            with (run_dir / "test_metrics.json").open("w", encoding="utf-8") as file:
                json.dump(metrics, file, ensure_ascii=False, indent=2)
            del model, test_loader
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    test_results = pd.DataFrame(rows)
    test_results.to_csv(OUTPUT_DIR / "final_test_results.csv", index=False)
    display(test_results)
    display(summarize_ablations(test_results).round(4))

    seed = SEEDS[0]
    fig, axes = plt.subplots(1, len(selected), figsize=(4.5 * len(selected), 3.8))
    axes = np.atleast_1d(axes)
    for ax, variant in zip(axes, selected):
        labels, predictions, _ = prediction_store[(variant, seed)]
        plot_confusion(labels, predictions, f"{variant} seed {seed}", ax)
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / "final_test_confusion_matrices.png",
        dpi=250,
        bbox_inches="tight",
    )
    plt.show()
    save_decision_report(results, test_results, best_variant)
    return test_results, best_variant


final_test_results = None
best_variant = None
if RUN_PIPELINE and DATA_AVAILABLE and ablation_results is not None:
    final_test_results, best_variant = run_final_test(ablation_results)
elif (OUTPUT_DIR / "final_test_results.csv").is_file():
    final_test_results = pd.read_csv(OUTPUT_DIR / "final_test_results.csv")
    if validation_best_variant is not None:
        best_variant = validation_best_variant
    display(final_test_results)
elif RUN_PIPELINE:
    print("数据未配置，最终测试未启动。")
else:
    print("RUN_PIPELINE=False，最终测试未启动。")

## 8. Attention 可视化

本次只运行三个 baseline，没有 LG-MAP attention；该单元保留为兼容旧报告并会自动跳过。

In [ ]:
@torch.inference_mode()
def visualize_attention(
    variant: str,
    seed: int = 42,
) -> None:
    if not DATA_AVAILABLE:
        print("未找到数据，跳过 attention 可视化。")
        return
    checkpoint = OUTPUT_DIR / variant / f"seed_{seed}" / "best_model.pt"
    if not checkpoint.is_file():
        print("未找到 checkpoint：", checkpoint)
        return
    if "LGMAP" not in variant:
        print("该模型没有 LG-MAP attention。")
        return

    model = load_trained_model(variant, str(checkpoint))
    _, val_loader, _ = make_loaders(seed, include_test=False)
    dataset = val_loader.dataset
    selected = {}
    for index in range(len(dataset)):
        image, label, name = dataset[index]
        if label not in selected:
            selected[label] = (image, name)
        if len(selected) == NUM_CLASSES:
            break
    if len(selected) != NUM_CLASSES:
        raise RuntimeError("验证集没有覆盖全部三个类别，无法绘制完整 attention 图")

    fig, axes = plt.subplots(3, 3, figsize=(10, 9))
    for label in range(NUM_CLASSES):
        image, name = selected[label]
        batch = image.unsqueeze(0).to(DEVICE)
        output = model(batch)
        original = (
            denormalize(batch).squeeze(0).permute(1, 2, 0).cpu().numpy()
        )
        height, width = output["attention_size"]
        attention = output["attention"].mean(dim=1).reshape(1, 1, height, width)
        attention = F.interpolate(
            attention,
            size=(IMAGE_HEIGHT, IMAGE_WIDTH),
            mode="bilinear",
            align_corners=False,
        ).squeeze().cpu().numpy()
        attention = (attention - attention.min()) / (
            attention.max() - attention.min() + 1e-8
        )

        mask = output["mask"]
        if mask is None:
            mask_image = np.ones((IMAGE_HEIGHT, IMAGE_WIDTH))
        else:
            mask_image = F.interpolate(
                mask,
                size=(IMAGE_HEIGHT, IMAGE_WIDTH),
                mode="bilinear",
                align_corners=False,
            ).squeeze().cpu().numpy()

        axes[label, 0].imshow(original)
        axes[label, 0].set_title(f"{CLASS_NAMES[label]}: {name}")
        axes[label, 1].imshow(original)
        axes[label, 1].imshow(mask_image, cmap="viridis", alpha=0.55)
        axes[label, 1].set_title("Soft colour prior")
        axes[label, 2].imshow(original)
        axes[label, 2].imshow(attention, cmap="jet", alpha=0.50)
        axes[label, 2].set_title("LG-MAP attention")
        for ax in axes[label]:
            ax.axis("off")

    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / f"attention_examples_{variant}_seed_{seed}.png",
        dpi=250,
        bbox_inches="tight",
    )
    plt.show()
    del model, val_loader


attention_variant = None
if ablation_summary is not None:
    lgmap_summary = ablation_summary.loc[
        ablation_summary["variant"].str.contains("LGMAP")
    ]
    if not lgmap_summary.empty:
        attention_variant = lgmap_summary.sort_values(
            ["macro_f1_mean", "n75_f1_mean", "ordinal_mae_mean"],
            ascending=[False, False, True],
        ).iloc[0]["variant"]

if attention_variant is not None:
    visualize_attention(attention_variant, SEEDS[0])
else:
    print("没有已完成的 LG-MAP 结果，跳过 attention 可视化。")

## 9. 一次运行后的判断

程序会把结论写入 `three_baselines_outputs/decision_report.md`。

采用以下固定规则，不再根据测试集反复改参数：

1. 验证集按平均 Macro-F1 排序，平分时依次比较 N75 F1 和 Ordinal MAE；
2. 三个 baseline 都进入最终测试，不因验证集结果跳过某个 baseline；
3. 同时报告 Accuracy、Macro-F1、N75 F1、Ordinal MAE 和 N0↔NFull 严重反向错误；
4. 无论结果正负，都保留这一套固定参数并如实报告，不再查看测试集调参。

## 参考

- Tan and Le (2019), EfficientNet:
  https://proceedings.mlr.press/v97/tan19a.html
- He et al. (2016), ResNet:
  https://openaccess.thecvf.com/content_cvpr_2016/html/He_Deep_Residual_Learning_CVPR_2016_paper.html
- Touvron et al. (2021), DeiT:
  https://arxiv.org/abs/2012.12877

## 10. 当前最终结果汇总（已写入 artifact）

> 本节是当前已经完成实验的汇总。Baseline 已完成 3 个 seed 的最终测试；RMOF-Net 7e 当前已有 seed 42 和 seed 52 的最终测试，seed 62 已完成训练/验证 checkpoint，但 final test 在中断前尚未写入。因此这里先报告 **current available results**，不是最终三 seed 完整版。

### 实验协议

- 数据划分固定为 `split_40_10_50.csv`：train `480`、validation `120`、test `600`，每类分别为 `160/40/200`。
- Baseline：EfficientNet-B0、ResNet18、DeiT-Tiny，每个模型 `seed 42/52/62`，结果来自 `three_baselines_outputs/final_test_results.csv`。
- 当前方法：`full_rmof_7e_mild_lr5e4_h25`，冻结对应 seed 的 B0 checkpoint，只训练 RMOF residual 分支，最多 `7` epoch，`mild` augmentation，LR `5e-4`，test 只在 validation-selected checkpoint 上评估一次。
- 当前方法训练时可训练参数为 `302,753`，总参数为 `4,314,144`。

### Final test results

| Model | Seeds | n | Accuracy | Macro-F1 | N75 F1 | Ordinal MAE | Endpoint confusions | Parameters |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| B0_Base | 42, 52, 62 | 3 | 0.6839 ± 0.0075 | 0.6767 ± 0.0077 | 0.5126 ± 0.0243 | 0.3339 ± 0.0075 | 10.6667 ± 0.5774 | 4,011,391 |
| ResNet18_Base | 42, 52, 62 | 3 | 0.6417 ± 0.0180 | 0.6370 ± 0.0145 | 0.4680 ± 0.0211 | 0.3739 ± 0.0135 | 9.3333 ± 2.8868 | 11,178,051 |
| DeiT_Tiny_Base | 42, 52, 62 | 3 | 0.6167 ± 0.0167 | 0.6128 ± 0.0144 | 0.4495 ± 0.0180 | 0.4111 ± 0.0151 | 16.6667 ± 1.5275 | 5,524,995 |
| RMOF-Net 7e (current) | 42, 52 | 2 | 0.6942 ± 0.0153 | 0.6869 ± 0.0201 | 0.5244 ± 0.0506 | 0.3217 ± 0.0165 | 9.5000 ± 0.7071 | 4,314,144 |

### RMOF seed status

| Seed | Val Macro-F1 | Best epoch | Completed epochs | Train samples | Final test |
| --- | --- | --- | --- | --- | --- |
| 42 | 0.6689 | 7 | 7 | 480 | done |
| 52 | 0.6553 | 0 | 7 | 480 | done |
| 62 | 0.6876 | 0 | 7 | 480 | pending |

### 当前结论

- 当前已有 test 结果下，RMOF-Net 7e 的平均 test Macro-F1 为 `0.6869`，高于最强 baseline EfficientNet-B0 的 `0.6767`。
- 相对 EfficientNet-B0，当前 RMOF-Net 7e 的 Macro-F1 提升约 `+0.0103`；Ordinal MAE 从 `0.3339` 降到 `0.3217`。
- 相对 ResNet18 和 DeiT-Tiny，RMOF-Net 7e 的 Macro-F1 分别高 `+0.0499` 和 `+0.0741`。
- 由于 RMOF seed 62 的 final test 尚未完成，最终论文/报告结论应在补齐 seed 62 后更新为完整 `mean ± std`。

### Final test comparison figures

![Final test Macro-F1 comparison](notebook_assets/final_test_macro_f1_comparison.png)

![Final test metric panel](notebook_assets/final_test_metric_panel.png)

### RMOF 7-epoch training curves

![RMOF 7e training curves](notebook_assets/rmof_7e_training_curves.png)

### Confusion matrices

Baseline final-test confusion matrices:

![Baseline confusion matrices](../three_baselines_outputs/final_test_confusion_matrices.png)

RMOF-Net 7e seed 42 final-test confusion matrix:

![RMOF seed 42 confusion matrix](../EfficientN/optimized_algorithm_outputs_v3/full_rmof_7e_mild_lr5e4_h25/seed_42/confusion_matrix.png)

RMOF-Net 7e seed 52 final-test confusion matrix:

![RMOF seed 52 confusion matrix](../EfficientN/optimized_algorithm_outputs_v3/full_rmof_7e_mild_lr5e4_h25/seed_52/confusion_matrix.png)

## 11. 当前消融结果

消融使用 validation set 做模块选择，不使用 test set。下表来自 `EfficientN/optimized_algorithm_outputs_v2/validation_module_results.csv`。

| Module | Run | Val Macro-F1 | Val Accuracy | Val N75 F1 | Val Ordinal MAE | Delta vs B0 |
| --- | --- | --- | --- | --- | --- | --- |
| Frozen B0 reference | B0_reference_validation | 0.6224 | 0.6333 | 0.3944 | 0.3750 | +0.0000 |
| Stage-4 residual | stage4_residual_lr3e4 | 0.6253 | 0.6333 | 0.4167 | 0.3833 | +0.0028 |
| Multi-scale regions | multiscale_regions_mild_lr3e4 | 0.6568 | 0.6583 | 0.5000 | 0.3500 | +0.0344 |
| Colour statistics | color_stats_mild_lr3e4 | 0.6395 | 0.6583 | 0.4179 | 0.3583 | +0.0170 |
| Colour texture | color_texture_mild_lr3e4 | 0.6396 | 0.6417 | 0.4810 | 0.3750 | +0.0172 |
| Scalar gate | scalar_gate_mild_lr3e4 | 0.6480 | 0.6500 | 0.4810 | 0.3583 | +0.0256 |
| Full RMOF-Net | full_rmof_mild_lr3e4 | 0.6696 | 0.6750 | 0.5067 | 0.3417 | +0.0472 |

![Validation ablation comparison](notebook_assets/ablation_validation_macro_f1.png)

### 消融结论

- Frozen B0 validation Macro-F1 为 `0.6224`。
- 单模块中，Multi-scale regions 提升最大，validation Macro-F1 达到 `0.6568`。
- Full RMOF-Net validation Macro-F1 达到 `0.6696`，相对 Frozen B0 提升 `+0.0472`。
- 当前短 epoch 配置 `full_rmof_7e_mild_lr5e4_h25` 在 7 epoch 内恢复了接近长训练最佳验证表现，同时避免 25 epoch 后段的过拟合。

